In [ ]:
# for colab users, you can download & upload the files with this code block.
# be sure to authenticate into google. do not modify this cell or the AG may break!

from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive_service = build('drive', 'v3')
file_ids = ['1UlsbmlJuiMgPg9pbyoCm11idqGS42_K1', '1_HB4CV_SaCJhUWINpQg7pMT9le0ttk7i']
file_names = ['hangman.py', 'hw1_word_counts_05.txt']
for file_id, file_name in zip(file_ids, file_names):
    request = drive_service.files().get_media(fileId=file_id)
    with open(file_name, "wb") as f:
        f.write(request.execute())

In [ ]:
from hangman import hangman_game

# Play the game by yourself
hangman_game()

In [ ]:
import random

def random_guess(letters_tried, word_pattern, word_counts):
    """Return a uniformly random letter A-Z that hasn't been tried yet."""
    alphabet = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
    remaining = sorted(alphabet - {l.upper() for l in letters_tried})
    return random.choice(remaining)

In [ ]:
def bayesian_guess(letters_tried, word_pattern, word_counts):
    """Return the letter most likely to appear in a blank position."""
    tried = {l.upper() for l in letters_tried}
    
    # Find all words in the corpus compatible with the evidence
    compatible = []
    for word, count in word_counts.items():
        ok = True
        for i, p in enumerate(word_pattern):
            if p != "_" and word[i] != p:
                ok = False
                break
            if p == "_" and word[i] in tried:
                ok = False
                break
        if ok:
            compatible.append((word, count))
    
    total = sum(c for _, c in compatible)
    if total == 0:
        return random_guess(letters_tried, word_pattern, word_counts)
    
    # For each candidate letter, compute P(letter in blank position | evidence)
    blank_positions = [i for i, p in enumerate(word_pattern) if p == "_"]
    best_letter = None
    best_score = -1.0
    for letter in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
        if letter in tried:
            continue
        score = 0.0
        for word, count in compatible:
            if any(word[i] == letter for i in blank_positions):
                score += count / total
        if score > best_score:
            best_score = score
            best_letter = letter
    return best_letter

In [ ]:
from typing import Optional, Callable
# from tqdm import tqdm

def benchmark(
    inference: Optional[Callable],
    num_runs: int = 1000,
    seed: int = 0
) -> None:
    '''
    Benchmark method for testing out the bayesian inference method. The parameters given are:

    - inference: The function that should match the bayesian_inference() method above.
    - seed: The seed to pass into the hangman_game function.

    Return type: none.

    Print the accuracy out of num_runs. If the function throws an error, it should count as a fail.
    '''
    # TODO: Implement benchmark testing for a given function
    # Hint: use `random.seed(seed)` ONCE at the start
    # Hint: pass `interactive = False` into the hangman_game function to run faster without outputs.
    # Optional: use the `tqdm` module to keep track of the evaluation progress.

benchmark(random_inference)
benchmark(bayesian_inference)